# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HimanshuSharma-2856/Flyrank-ml--internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: the validation boundary changes the result

The paper reports an earlier nine-feature random-forest demonstration at `0.996` with random rows and `0.496` with client grouping. My constructive methodology question is: **does the grouped result represent the deployment question better because complete clients are held out, and are the two numbers clearly labelled as separate experiments rather than treated as one before/after model result?** I would preserve the impressive random-row number as a warning about memorisation, not as evidence of generalisation.

### Finding 2: a higher mean is not a win in every fold

The paper reports HistGradientBoosting at `0.504 ± 0.211` precision@50 versus the rule at `0.384 ± 0.182`, with the model higher in 3 folds and lower in 2. My constructive methodology question is: **does the mean comparison show the fold spread, base rates, and per-fold wins clearly enough for a reader to judge stability, and is the claim limited to this grouped evaluation rather than implying universal improvement?** I would report the direction and variability beside the mean and keep the result as measured decision-support evidence.

These questions guide my own audit: I will compare random rows with a client-grouped holdout, print the test base rate, test label-derived features explicitly, and inspect real false positives and false negatives.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
here = Path.cwd()
candidates = [here, *here.parents, Path("/workspaces/Flyrank-ml--internship"), Path("/workspace")]
root_matches = [path for path in candidates if (path / "data" / "raw" / "content_refresh_anonymized.csv").exists()]
if not root_matches:
    local_data_matches = Path.home().glob("Downloads/**/data/raw/content_refresh_anonymized.csv")
    root_matches = [data_path.parents[2] for data_path in local_data_matches]
if not root_matches:
    raise FileNotFoundError(f"Could not locate data from notebook cwd: {here}")
ROOT = root_matches[0]
raw = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
raw["is_declining_label"] = raw["trend_direction"].fillna("").str.lower().eq("down").astype(int)
for column in raw.select_dtypes(include=["number"]).columns:
    raw[column] = raw[column].replace([np.inf, -np.inf], np.nan)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d",
    "clicks_90d", "sessions_90d", "ai_sessions_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
numeric_features = [column for column in numeric_features if column in raw.columns]
categorical_features = [column for column in categorical_features if column in raw.columns]
feature_columns = numeric_features + categorical_features

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE,
    )),
])

def precision_at_k(labels, scores, k=50):
    scored = pd.DataFrame({"label": np.asarray(labels), "score": np.asarray(scores)})
    return float(scored.sort_values("score", ascending=False).head(min(k, len(scored)))["label"].mean())

def baseline_score(frame):
    def percentile(series):
        return pd.to_numeric(series, errors="coerce").fillna(0).rank(method="average", pct=True)
    values = frame.copy()
    impressions = percentile(np.log1p(values["impressions_90d"]))
    freshness = percentile(values["days_since_last_update"])
    avg_position = values["avg_position"].fillna(0)
    position = (1 - avg_position.clip(lower=1, upper=50).rank(pct=True)) * impressions * (avg_position > 0)
    depth = (1 - percentile(values["word_count"])) * impressions
    return (0.40 * impressions + 0.30 * freshness + 0.25 * position + 0.05 * depth).clip(0, 1)

raw["baseline_score"] = baseline_score(raw)
assert raw["is_declining_label"].nunique() == 2
print(f"Rows: {len(raw):,}; clients: {raw['client_id'].nunique():,}; positive base rate: {raw['is_declining_label'].mean():.3f}")
print(f"Features: {len(feature_columns)}; target: is_declining_label")

Rows: 30,000; clients: 32; positive base rate: 0.542
Features: 26; target: is_declining_label


## 2. My model under an honest split (before/after)

The earlier Week-5 model used a client-grouped holdout, which is better aligned with the question “does this transfer to an unseen client?” To make the improvement visible, I will compare the same Random Forest and the same Precision@50 metric under a stratified random-row split and a client-grouped split. The random-row score is not a production claim; it is a diagnostic for how much the validation boundary matters. Both evaluations print the test base rate.

In [2]:
def evaluate_split(train_indices, test_indices, split_name):
    train_frame = raw.iloc[train_indices]
    test_frame = raw.iloc[test_indices]
    split_model = Pipeline(model.steps)
    split_model.fit(train_frame[feature_columns], train_frame["is_declining_label"])
    scores = split_model.predict_proba(test_frame[feature_columns])[:, 1]
    baseline_scores = test_frame["baseline_score"].to_numpy()
    result = {
        "split": split_name,
        "train_rows": len(train_frame),
        "test_rows": len(test_frame),
        "test_clients": test_frame["client_id"].nunique(),
        "test_base_rate": test_frame["is_declining_label"].mean(),
        "baseline_precision@50": precision_at_k(test_frame["is_declining_label"], baseline_scores),
        "model_precision@50": precision_at_k(test_frame["is_declining_label"], scores),
        "model_average_precision": average_precision_score(test_frame["is_declining_label"], scores),
    }
    return result, split_model, scores, test_frame

all_indices = np.arange(len(raw))
random_train, random_test = train_test_split(
    all_indices, test_size=0.20, random_state=RANDOM_STATE, stratify=raw["is_declining_label"]
)
clients = raw["client_id"].fillna("unknown").astype(str).drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_clients = set(rng.permutation(clients)[:max(1, round(len(clients) * 0.20))])
group_test_mask = raw["client_id"].fillna("unknown").astype(str).isin(test_clients).to_numpy()
group_train = all_indices[~group_test_mask]
group_test = all_indices[group_test_mask]

random_result, random_model, random_scores, random_test_frame = evaluate_split(random_train, random_test, "random_row_holdout")
group_result, group_model, group_scores, group_test_frame = evaluate_split(group_train, group_test, "client_grouped_holdout")
comparison = pd.DataFrame([random_result, group_result]).set_index("split")
print(comparison.round(3).to_string())
print(f"\nPrecision@50 gap (random minus grouped): {comparison.loc['random_row_holdout', 'model_precision@50'] - comparison.loc['client_grouped_holdout', 'model_precision@50']:.3f}")
assert comparison["test_base_rate"].between(0, 1).all()
assert set(group_train).isdisjoint(set(group_test))
assert set(raw.iloc[group_train]["client_id"]).isdisjoint(set(raw.iloc[group_test]["client_id"]))
assert comparison.loc["client_grouped_holdout", "model_precision@50"] >= 0

                        train_rows  test_rows  test_clients  test_base_rate  baseline_precision@50  model_precision@50  model_average_precision
split                                                                                                                                          
random_row_holdout           24000       6000            31           0.542                   0.48                0.94                    0.767
client_grouped_holdout       27675       2325             6           0.391                   0.28                0.66                    0.610

Precision@50 gap (random minus grouped): 0.280


## 3. Leakage audit

I checked three leakage categories. First, `trend_direction` and `trend_pct` are label-derived because the target is defined from the trend direction. Second, the 90-day activity features overlap the 30-day trend window in this starter snapshot, so they are useful for this teaching comparison but cannot support a clean time-forward claim. Third, `baseline_score` is a decision-derived feature and is used only for comparison, never as a model input. The grouped split reduces client memorisation but does not solve the temporal overlap limitation.

In [3]:
forbidden_features = {
    "trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id",
    "baseline_score", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
assert not forbidden_features.intersection(feature_columns)
print("Static leakage audit: PASS")
print(f"Forbidden fields checked: {sorted(forbidden_features)}")
print("Temporal limitation: 90-day aggregates overlap the 30-day label construction window; a future time-forward dataset is needed for a deployment claim.")

# Deliberate label-derived probe: diagnostic only, never retained as a feature.
leaky_features = ["trend_pct"]
leaky_preprocessor = ColumnTransformer([
    ("leaky", SimpleImputer(strategy="median"), leaky_features),
])
leaky_model = Pipeline([
    ("preprocessor", leaky_preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)),
])
leaky_model.fit(raw.iloc[group_train][leaky_features], raw.iloc[group_train]["is_declining_label"])
leaky_scores = leaky_model.predict_proba(raw.iloc[group_test][leaky_features])[:, 1]
print(f"Deliberate leaky trend_pct probe Precision@50: {precision_at_k(raw.iloc[group_test]['is_declining_label'], leaky_scores):.3f}")
print("The probe demonstrates why label-derived fields are forbidden; trend_pct remains excluded from the honest model.")

Static leakage audit: PASS
Forbidden fields checked: ['baseline_score', 'clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'is_declining_label', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']
Temporal limitation: 90-day aggregates overlap the 30-day label construction window; a future time-forward dataset is needed for a deployment claim.
Deliberate leaky trend_pct probe Precision@50: 1.000
The probe demonstrates why label-derived fields are forbidden; trend_pct remains excluded from the honest model.


## 4. Failure examples and claim rewrite

The grouped result is useful but imperfect. I will inspect the highest-confidence false positives and lowest-confidence false negatives on the grouped holdout, then rewrite the original model claim so it says exactly what this notebook measured.

In [4]:
group_errors = group_test_frame[[
    "content_id", "client_id", "content_type", "is_declining_label",
    "impressions_90d", "clicks_90d", "days_since_last_update", "avg_position",
]].copy()
group_errors["score"] = group_scores
group_errors["predicted_label"] = (group_scores >= 0.5).astype(int)
false_positives = group_errors[(group_errors["predicted_label"] == 1) & (group_errors["is_declining_label"] == 0)].sort_values("score", ascending=False)
false_negatives = group_errors[(group_errors["predicted_label"] == 0) & (group_errors["is_declining_label"] == 1)].sort_values("score", ascending=True)
print("Grouped-holdout false positives:")
print(false_positives.head(3).to_string(index=False))
print("\nGrouped-holdout false negatives:")
print(false_negatives.head(3).to_string(index=False))

print("\nOriginal claim: The Random Forest model identifies declining pages and beats the baseline.")
print("Rewritten claim: On this starter snapshot, the Random Forest produced higher observed Precision@50 than the baseline on both a random-row holdout (0.94 vs 0.48) and a client-grouped holdout (0.66 vs 0.28). The grouped result is the more honest transfer check, but the 90-day features overlap the 30-day label construction window, so this is directional decision-support evidence rather than a time-forward production claim.")
assert len(false_positives) > 0
assert len(false_negatives) > 0

Grouped-holdout false positives:
          content_id         client_id    content_type  is_declining_label  impressions_90d  clicks_90d  days_since_last_update  avg_position    score  predicted_label
content_331182ca4cae client_f74efabef1 keyword article                   0             3026           0                      20          35.9 0.760562                1
content_d2dffcc697a4 client_f74efabef1 keyword article                   0             5091          10                      20          14.1 0.745269                1
content_643f585dc7f7 client_f74efabef1 keyword article                   0              761           3                      20          25.1 0.740013                1

Grouped-holdout false negatives:
          content_id         client_id    content_type  is_declining_label  impressions_90d  clicks_90d  days_since_last_update  avg_position    score  predicted_label
content_34b14c00f80c client_d4735e3a26  feedly article                   1                3  

## 5. Self-check

- [x] Two paper findings are named with constructive methodology questions.
- [x] Random-row and client-grouped model results are compared with base rates.
- [x] Leakage audit excludes label-derived, ID, decision-derived, and overlapping-window fields.
- [x] A deliberate `trend_pct` probe demonstrates why label-derived features are forbidden.
- [x] Grouped false positives and false negatives are shown.
- [x] The model claim is rewritten with observed, directional, and decision-support language.
- [ ] Run every cell top to bottom in a clean kernel, inspect outputs, save, commit, and submit the repository URL.

The final honest conclusion is limited: the model ranks this starter snapshot better than the hand-built baseline on the measured holdouts, but the random-row result is optimistic and the starter snapshot does not provide a clean future time-forward test.